In [2]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [3]:
%load_ext autoreload
%autoreload 2

In [6]:
import os

In [4]:
import numpy as np

def apply_gaussian_noise(data, sigma=0.01):
    """
    Apply Gaussian noise to the input data.

    Parameters:
    - data: np.ndarray or pd.DataFrame, input data to perturb.
    - sigma: float, standard deviation of the Gaussian noise.

    Returns:
    - Perturbed data with added Gaussian noise.
    """
    noise = np.random.normal(0, sigma, data.shape)
    return data + noise


In [5]:
#Update the generate_cfs Method
def generate_cfs(dataset, continuous_features, model, model_backend, perturbation_method, target_name, sigma=0.01):
    # Split the data
    train_dataset, test_dataset, _, _ = split_data(dataset, target_name)

    x_train = train_dataset.drop(target_name, axis=1)
    x_test = test_dataset.drop(columns=[target_name])

    # Apply Gaussian noise to the test input
    x_test_original = x_test[1:2]
    x_test_perturbed = apply_gaussian_noise(x_test_original, sigma=sigma)

    # Generate counterfactuals for original and perturbed inputs
    d = dice_ml_x.Data(dataframe=train_dataset, continuous_features=continuous_features, outcome_name=target_name)
    m = dice_ml_x.Model(model=model, backend=model_backend)

    exp = dice_ml_x.DiceX(d, m, method='genetic' if model_backend == 'sklearn' else 'gradient')

    cfs_original = exp.generate_counterfactuals(x_test_original, total_CFs=4, desired_class="opposite", perturbation_method=perturbation_method)
    cfs_perturbed = exp.generate_counterfactuals(x_test_perturbed, total_CFs=4, desired_class="opposite", perturbation_method=perturbation_method)

    # Compute stability metric
    stability = self.compute_stability(cfs_original.to_dataframe(), cfs_perturbed.to_dataframe())

    return {
        "cfs_original": cfs_original.to_dataframe(),
        "cfs_perturbed": cfs_perturbed.to_dataframe(),
        "stability": stability,
    }

def compute_stability(self, cfs_original, cfs_perturbed):
    """
    Compute stability metric as the average Euclidean distance between
    counterfactuals generated from original and perturbed inputs.

    Parameters:
    - cfs_original: DataFrame, counterfactuals for original input.
    - cfs_perturbed: DataFrame, counterfactuals for perturbed input.

    Returns:
    - Stability score (float).
    """
    distances = np.linalg.norm(cfs_original.values - cfs_perturbed.values, axis=1)
    return np.mean(distances)


In [ ]:
#Update load_and_train Method
def load_and_train(self, batch_size, artefact_path=None, sigma=0.01):
    num_processes = len(self.datasets) * len(self.backends)

    if artefact_path is None:
        artefact_path = "benchmarking_artefact"
    if not os.path.isdir(artefact_path):
        os.mkdir(artefact_path)

    with tqdm(total=num_processes, desc="Benchmarking", leave=True) as d_pbar:
        for dataset_name in self.datasets:
            df, target_column = self.load_dataset(dataset_name)

            continuous_features = df.select_dtypes(include=[np.number]).columns.to_list()
            continuous_features.remove(target_column)
            self.results[dataset_name] = {}

            for backend in self.backends:
                # Preprocess data and train model
                x_train_transformed, x_test_transformed, train_df, test_df, y_train, y_test = self.preprocess_data(
                    backend=backend, df=df, continuous_features=continuous_features, target_name=target_column, batch_size=batch_size
                )

                model = self.train_model(backend, x_train_transformed, x_test_transformed, y_train)

                # Evaluate accuracy
                if backend == "sklearn":
                    accuracy = model.score(x_test_transformed, y_test)
                elif backend == "PYT":
                    accuracy = self.compute_pytorch_accuracy(model, x_test_transformed)
                elif backend == "TF2":
                    accuracy = self.compute_keras_accuracy(model, x_test_transformed)

                backend_results = {
                    "accuracy": accuracy,
                    "cfs": {},
                    "stability": {},
                    "input_instance": {},
                    "time": {},
                }

                # Generate counterfactuals and evaluate stability
                for method in self.perturbation_methods:
                    cfs_data = self.generate_cfs(
                        df, continuous_features, model, backend, method, target_column, sigma
                    )
                    backend_results["cfs"][method] = cfs_data["cfs_original"]
                    backend_results["stability"][method] = cfs_data["stability"]
                    backend_results["time"][method] = cfs_data["time"]
                    self.results[dataset_name][backend] = backend_results

                d_pbar.update(1)


In [ ]:
from dice_ml_x.benchmarking import Benchmarking

# Datasets for Benchmarking
datasets = ["compas-recidivism", "adult-income", "lending-club", "german-credit"]

# Backends for Model Training
backends = ["sklearn", "PYT", "TF2"]

# Perturbation Methods for Counterfactual Explanations
methods = ['gaussian', 'random', 'spherical']

# Initialize the Benchmarking Framework
benchmarking = Benchmarking(
    datasets=datasets,
    backends=backends,
    perturbation_methods=methods
)

# Load datasets, train models, and evaluate counterfactuals
benchmarking.load_and_train(batch_size=16)

# Access Results
results = benchmarking.results

# Example: Inspecting Results for a Dataset and Backend
for dataset in datasets:
    for backend in backends:
        if dataset in results and backend in results[dataset]:
            backend_results = results[dataset][backend]
            print(f"Results for {dataset} using {backend}:")
            print("Accuracy:", backend_results.get("accuracy"))
            for method in methods:
                print(f"Method: {method}")
                print("Counterfactuals:", backend_results["cfs"].get(method))
                print("Stability:", backend_results["stability"].get(method))
                print("Generation Time:", backend_results["time"].get(method))
